# Pipeline de asistencia: Excel → SQL Server

Este cuaderno toma los reportes **"Inasistencias por curso"** (.xlsx), los limpia y los carga en la tabla `dbo.asistencia`.

| Paso | Qué hace |
|---|---|
| 0 | Instalar librerías (solo la primera vez) |
| 1 | Configuración: carpeta de los Excel y datos del servidor |
| 2 | Importar librerías |
| 3 | Mirar un archivo "crudo" para entender su forma |
| 4 | Leer los metadatos del encabezado (docente, programa…) |
| 5 | Sacar el código del curso del nombre del archivo |
| 6 | Leer la grilla de estudiantes × fechas |
| 7 | Limpiar: documentos, fechas en español y estados |
| 8 | Pasar de formato **ancho** (una columna por fecha) a **largo** (una fila por fecha) |
| 9 | Validar antes de cargar |
| 10 | Conectarse a SQL Server |
| 11 | Cargar con `MERGE` (inserta lo nuevo, actualiza lo que cambió) |
| 12 | Procesar **todos** los archivos de la carpeta |
| 13 | Verificar en la base de datos |

> **Cómo usarlo:** ejecuta las celdas en orden con **Shift + Enter**. Del paso 3 al 9 se trabaja con **un solo archivo** para que veas qué pasa en cada etapa; en el paso 12 se aplica lo mismo a todos.

## Paso 0 · Instalar librerías (solo la primera vez)

- `pandas`: manejar tablas en Python.
- `openpyxl`: leer archivos .xlsx.
- `sqlalchemy` + `pyodbc`: conectarse a SQL Server.

También necesitas el **ODBC Driver 18 for SQL Server** instalado en Windows (se descarga de la página de Microsoft). Quita el `#` de la siguiente celda, ejecútala una vez y vuelve a ponerlo.

In [1]:
# %pip install pandas openpyxl sqlalchemy pyodbc

## Paso 1 · Configuración

**Esta es la única celda que tienes que modificar.**

- `CARPETA`: dónde están los Excel. La `r` antes de las comillas evita problemas con las `\` de Windows.
- `SERVIDOR` y `BASE`: tu SQL Server. Si entras con tu usuario de Windows, deja `USUARIO = None`.
- `CARGAR_EN_BD`: déjalo en `False` mientras pruebas. Con `False` el cuaderno hace todo **menos** escribir en la base.

In [3]:
CARPETA  = r"C:\Users\sebastian.mendez\OneDrive - CENTRO COLOMBO AMERICANO\Escritorio\Automatizacion Oxford\Descargas\Q10\inasistencias"      # <- cambia esto
SERVIDOR = "bi-db-group.database.windows.net"
BASE     = "rendimiento_academico"
USUARIO  = "bi_admin"
CLAVE    = "53/O~D&2m#o}"
DRIVER   = "ODBC Driver 18 for SQL Server"

CARGAR_EN_BD = True

## Paso 2 · Importar librerías

`re` sirve para buscar patrones en texto (expresiones regulares) y `unicodedata` para manejar tildes.

In [4]:
import re
import unicodedata
from datetime import date, datetime
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook

pd.set_option("display.max_columns", 20)

archivos = sorted(Path(CARPETA).glob("*.xlsx"))
print(f"Encontré {len(archivos)} archivos:")
for a in archivos:
    print("  -", a.name)

Encontré 14 archivos:
  - 2026-10-21_1548_Inasistencias_por_curso_B2.1-7PM215_SDDE_2026-2_B2G_YES_BOGOTA-2_C1_SEP_2026_.xlsx
  - 2026-10-21_1548_Inasistencias_por_curso_B2.1-7PM216_SDDE_2026-2_B2G_YES_BOGOTA-2_C1_SEP_2026_.xlsx
  - 2026-10-21_1548_Inasistencias_por_curso_C1.1-7PM217_SDDE_2026-2_B2G_YES_BOGOTA-2_C1_SEP_2026_.xlsx
  - 2026-10-21_1549_Inasistencias_por_curso_B2.1-6AM60_SDDE_2026-2_B2G_YES_BOGOTA-2_C1_SEP_2026_.xlsx
  - 2026-10-21_1549_Inasistencias_por_curso_B2.1-7PM205_SDDE_2026-2_B2G_YES_BOGOTA-2_C1_SEP_2026_.xlsx
  - 2026-10-21_1549_Inasistencias_por_curso_B2.1-7PM206_SDDE_2026-2_B2G_YES_BOGOTA-2_C1_SEP_2026_.xlsx
  - 2026-10-21_1549_Inasistencias_por_curso_B2.1-7PM207_SDDE_2026-2_B2G_YES_BOGOTA-2_C1_SEP_2026_.xlsx
  - 2026-10-21_1549_Inasistencias_por_curso_B2.1-7PM208_SDDE_2026-2_B2G_YES_BOGOTA-2_C1_SEP_2026_.xlsx
  - 2026-10-21_1549_Inasistencias_por_curso_B2.1-7PM209_SDDE_2026-2_B2G_YES_BOGOTA-2_C1_SEP_2026_.xlsx
  - 2026-10-21_1549_Inasistencias_por_curso_B2.1-7PM

## Paso 3 · Mirar un archivo crudo

Antes de limpiar hay que **entender la forma** del archivo. Tomamos el primero y mostramos sus primeras filas tal cual vienen.

Fíjate en que:
- Las filas 1 a 4 son **metadatos** (sede, programa, docente, fechas del curso).
- La fila 6 es el **encabezado**: `Estudiantes`, `Identificación` y luego **una columna por fecha**.
- Después vienen los estudiantes, y al final hay un **GLOSARIO** que no son datos.

In [5]:
archivo = archivos[0]
print("Archivo de prueba:", archivo.name)

wb = load_workbook(archivo, data_only=True, read_only=True)
hoja = wb.worksheets[0]
filas = [list(f) for f in hoja.iter_rows(values_only=True)]

print("Total de filas:", len(filas))
pd.DataFrame(filas).iloc[:10, :10]      # primeras 10 filas y 10 columnas

Archivo de prueba: 2026-10-21_1548_Inasistencias_por_curso_B2.1-7PM215_SDDE_2026-2_B2G_YES_BOGOTA-2_C1_SEP_2026_.xlsx
Total de filas: 41


,0,1,2,3,4,5,6,7,8,9
0,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,None,Sede - jornada:,NaN,B2G Única,NaN,Programa:,YES BOGOTA 2026-2,NaN,NaN,NaN
2,None,Período:,NaN,B2G_YES BOGOTA-2 C1_SEP_2026,NaN,Asignatura:,SDDE26-2_B2-1,NaN,NaN,NaN
3,None,Docente:,NaN,Pedreros Cely Sergio Alejandro,NaN,Fecha Inicio:,7/09/2026,NaN,Fecha Fin:,13/11/2026
4,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,None,Estudiantes,NaN,NaN,Identificación,sept. 07 2026,sept. 08 2026,sept. 09 2026,sept. 10 2026,sept. 11 2026
6,None,Abril Fonseca Manuel Fabian,NaN,NaN,C.C. 1015455378,A,A,A,A,A
7,None,Alfonso Rodriguez Samuel Esteban,NaN,NaN,C.C. 1010964859,A,A,A,A,A
8,None,Arevalo Linares Juan Camilo,NaN,NaN,C.C. 1034518574,A,A,A,A,A
9,None,Ariza Juan,NaN,NaN,C.C. 1000381858,A,A,A,A,A


## Paso 4 · Leer los metadatos del encabezado

Los metadatos vienen como **"Etiqueta:" + valor** a su derecha (por ejemplo `Docente:` → `Moreno Quiroga Sergio David`).

En lugar de leer celdas fijas como `D4`, **buscamos la etiqueta**. Así, si mañana la plantilla mueve una columna, el código sigue funcionando.

Primero definimos dos funciones pequeñas de limpieza de texto que usaremos en todo el cuaderno.

In [6]:
def limpiar_texto(valor):
    # Quita espacios dobles, espacios raros y devuelve None si queda vacío
    if valor is None:
        return None
    texto = unicodedata.normalize("NFC", str(valor)).replace("\xa0", " ")
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto or None

def sin_tildes(texto):
    # 'Período' -> 'Periodo' (para comparar etiquetas sin problemas)
    return "".join(c for c in unicodedata.normalize("NFD", texto)
                   if unicodedata.category(c) != "Mn")

print(limpiar_texto("  Moreno   Quiroga  "))
print(sin_tildes("Período"))

Moreno Quiroga
Periodo


In [7]:
# Etiqueta en el Excel -> nombre de columna en nuestra tabla
ETIQUETAS = {
    "sede - jornada": "sede_jornada",
    "programa":       "programa",
    "periodo":        "periodo",
    "asignatura":     "asignatura",
    "docente":        "docente",
    "fecha inicio":   "fecha_inicio",
    "fecha fin":      "fecha_fin",
}

def leer_metadatos(filas):
    meta = {}
    for fila in filas:
        # Paramos al llegar a la fila de encabezado de la grilla
        if any(limpiar_texto(c) == "Estudiantes" for c in fila):
            break
        for j, celda in enumerate(fila):
            texto = limpiar_texto(celda)
            if texto and texto.endswith(":"):
                clave = ETIQUETAS.get(sin_tildes(texto[:-1]).lower())
                if clave:
                    # el valor es la siguiente celda con contenido a la derecha
                    valor = next((v for v in fila[j + 1:] if limpiar_texto(v)), None)
                    meta[clave] = limpiar_texto(valor)
    return meta

meta = leer_metadatos(filas)
meta

{'sede_jornada': 'B2G Única',
 'programa': 'YES BOGOTA 2026-2',
 'periodo': 'B2G_YES BOGOTA-2 C1_SEP_2026',
 'asignatura': 'SDDE26-2_B2-1',
 'docente': 'Pedreros Cely Sergio Alejandro',
 'fecha_inicio': '7/09/2026',
 'fecha_fin': '13/11/2026'}

## Paso 5 · Sacar el código del curso del nombre del archivo

**Dato importante:** dos de tus archivos tienen la **misma asignatura** (`SDDE26-2_B2-1`) pero son **grupos distintos**, con docentes distintos. Lo único que los diferencia es el código que viene en el **nombre del archivo**: `B2.1-7PM216` y `B2.1-6AM58`.

Por eso lo sacamos del nombre con una expresión regular: buscamos el texto que va después de `Inasistencias_por_curso_` y antes del siguiente `_`.

In [8]:
def codigo_desde_nombre(nombre_archivo):
    m = re.search(r"Inasistencias_por_curso_([^_]+)_", nombre_archivo)
    if not m:
        raise ValueError(f"No encontré el código del curso en: {nombre_archivo}")
    return m.group(1)

def fecha_reporte_desde_nombre(nombre_archivo):
    # '2026-10-16_0817_...' -> 2026-10-16 08:17
    m = re.search(r"(\d{4}-\d{2}-\d{2})_(\d{4})_", nombre_archivo)
    return datetime.strptime(m.group(1) + m.group(2), "%Y-%m-%d%H%M") if m else None

codigo_curso  = codigo_desde_nombre(archivo.name)
fecha_reporte = fecha_reporte_desde_nombre(archivo.name)
print("Curso:", codigo_curso)
print("Reporte generado:", fecha_reporte)

Curso: B2.1-7PM215
Reporte generado: 2026-10-21 15:48:00


## Paso 6 · Leer la grilla de estudiantes × fechas

1. Buscamos la fila de encabezado (la que contiene `Estudiantes`).
2. Ubicamos cada columna **por su título**, no por su letra.
3. Leemos estudiantes hasta encontrar una fila vacía o el `GLOSARIO`.

Por ahora los encabezados de fecha siguen siendo texto (`'sept. 07 2026'`); los convertimos en el paso 7.

In [9]:
# 1) fila de encabezado
fila_enc = next(i for i, f in enumerate(filas)
                if any(limpiar_texto(c) == "Estudiantes" for c in f))
encabezado = [limpiar_texto(c) for c in filas[fila_enc]]

# 2) posición de cada columna
col_nombre = encabezado.index("Estudiantes")
col_doc    = encabezado.index("Identificación")
col_asis   = encabezado.index("ASIST.ACUM")
col_inas   = encabezado.index("INAS.ACUM")

# 3) filas de estudiantes
registros = []
for fila in filas[fila_enc + 1:]:
    nombre = limpiar_texto(fila[col_nombre])
    if not nombre or nombre.upper() == "GLOSARIO":
        break
    registros.append(fila)

print("Fila de encabezado (índice):", fila_enc)
print("Estudiantes encontrados:", len(registros))
print("Ejemplo de encabezados:", encabezado[col_doc:col_doc + 4], "...", encabezado[-2:])

Fila de encabezado (índice): 5
Estudiantes encontrados: 30
Ejemplo de encabezados: ['Identificación', 'sept. 07 2026', 'sept. 08 2026', 'sept. 09 2026'] ... ['ASIST.ACUM', 'INAS.ACUM']


## Paso 7 · Limpiar los datos

Hay tres cosas que vienen "sucias":

| Dato | Viene así | Lo dejamos así |
|---|---|---|
| Documento | `'C.C. 1122919082'` | `tipo = 'CC'`, `numero = '1122919082'` (texto, para no perder ceros) |
| Fecha del encabezado | `'sept. 07 2026'` | `2026-09-07` (tipo fecha) |
| Fecha del curso | `'7/09/2026'` (día/mes/año) | `2026-09-07` |
| Estado | `'A'`, `'I'`, `''` | `'A'`, `'I'`, `'I*'` en mayúscula; el **vacío se descarta** |

**¿Por qué descartar el vacío?** Una celda vacía **no es una falta**: significa que esa clase todavía no se ha registrado (son fechas futuras).

In [10]:
TIPOS_DOC = {"CC": "CC", "TI": "TI", "CE": "CE", "PPT": "PPT", "PEP": "PEP",
             "PA": "PA", "PAS": "PA", "RC": "RC", "NIT": "NIT"}

def separar_documento(valor):
    # 'C.C. 1122919082' -> ('CC', '1122919082')
    texto = limpiar_texto(valor)
    m = re.fullmatch(r"([A-Za-z.\s]+?)\s*([0-9A-Za-z-]+)", texto or "")
    if not m:
        raise ValueError(f"Documento con formato raro: {valor!r}")
    tipo   = re.sub(r"[^A-Z]", "", m.group(1).upper())      # 'C.C.' -> 'CC'
    numero = re.sub(r"[^0-9A-Za-z]", "", m.group(2)).upper()
    if tipo not in TIPOS_DOC:
        raise ValueError(f"Tipo de documento desconocido: {valor!r}")
    return TIPOS_DOC[tipo], numero

print(separar_documento("C.C. 1122919082"))

('CC', '1122919082')


In [11]:
MESES = {"ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6, "jul": 7,
         "ago": 8, "sep": 9, "sept": 9, "set": 9, "oct": 10, "nov": 11, "dic": 12}

def fecha_encabezado(valor):
    # 'sept. 07 2026' -> date(2026, 9, 7). Devuelve None si no es una fecha.
    if isinstance(valor, datetime):
        return valor.date()
    texto = sin_tildes(limpiar_texto(valor) or "").lower()
    m = re.fullmatch(r"([a-z]+)\.?\s+(\d{1,2})\s+(\d{4})", texto)
    if not m or m.group(1) not in MESES:
        return None
    return date(int(m.group(3)), MESES[m.group(1)], int(m.group(2)))

def fecha_dia_mes_anio(valor):
    # '7/09/2026' -> date(2026, 9, 7)
    if isinstance(valor, datetime):
        return valor.date()
    return datetime.strptime(limpiar_texto(valor), "%d/%m/%Y").date()

print(fecha_encabezado("sept. 07 2026"))
print(fecha_encabezado("ASIST.ACUM"))      # no es fecha -> None
print(fecha_dia_mes_anio("7/09/2026"))

2026-09-07
None
2026-09-07


In [12]:
# Qué columnas son fechas: {posición_columna: fecha}
columnas_fecha = {j: f for j, h in enumerate(encabezado) if (f := fecha_encabezado(h))}

fecha_inicio = fecha_dia_mes_anio(meta["fecha_inicio"])
fecha_fin    = fecha_dia_mes_anio(meta["fecha_fin"])

print("Columnas de fecha:", len(columnas_fecha))
print("Primera:", min(columnas_fecha.values()), "| Última:", max(columnas_fecha.values()))
print("Curso va del", fecha_inicio, "al", fecha_fin)

Columnas de fecha: 48
Primera: 2026-09-07 | Última: 2026-11-13
Curso va del 2026-09-07 al 2026-11-13


## Paso 8 · De formato ancho a formato largo

Este es **el paso más importante**. El Excel tiene **una columna por fecha**:

| Estudiante | 07-sep | 08-sep | 09-sep | … |
|---|---|---|---|---|
| Acosta | A | A | I | … |

La tabla en SQL Server tiene **una fila por fecha**:

| Estudiante | fecha | estado |
|---|---|---|
| Acosta | 2026-09-07 | A |
| Acosta | 2026-09-08 | A |
| Acosta | 2026-09-09 | I |

Recorremos cada estudiante y, dentro de él, cada columna de fecha. Por cada celda **con valor** creamos una fila.

De paso guardamos los acumulados del Excel (`ASIST.ACUM` e `INAS.ACUM`): **no** van a la base, pero los usamos en el paso 9 para comprobar que no perdimos nada.

In [13]:
filas_largas = []
acumulados = []

for fila in registros:
    tipo, numero = separar_documento(fila[col_doc])
    nombre = limpiar_texto(fila[col_nombre])
    acumulados.append({"tipo_documento": tipo, "numero_documento": numero,
                       "asist_acum": fila[col_asis], "inas_acum": fila[col_inas]})

    for j, fecha in columnas_fecha.items():
        estado = (limpiar_texto(fila[j]) or "").upper()
        if not estado:                 # vacío = aún no registrado -> se omite
            continue
        filas_largas.append({
            "curso_codigo":     codigo_curso,
            "asignatura":       meta["asignatura"],
            "docente":          meta["docente"],
            "programa":         meta["programa"],
            "periodo":          meta["periodo"],
            "sede_jornada":     meta["sede_jornada"],
            "tipo_documento":   tipo,
            "numero_documento": numero,
            "nombre_completo":  nombre,
            "fecha":            fecha,
            "estado":           estado,
            "archivo_origen":   archivo.name,
        })

df = pd.DataFrame(filas_largas)
df_acum = pd.DataFrame(acumulados)

print(f"{len(registros)} estudiantes × {len(columnas_fecha)} fechas = "
      f"{len(registros) * len(columnas_fecha)} celdas")
print(f"Filas con marca (las que se cargan): {len(df)}")
df.head()

30 estudiantes × 48 fechas = 1440 celdas
Filas con marca (las que se cargan): 930


,curso_codigo,asignatura,docente,programa,periodo,sede_jornada,tipo_documento,numero_documento,nombre_completo,fecha,estado,archivo_origen
0,B2.1-7PM215,SDDE26-2_B2-1,Pedreros Cely Sergio Alejandro,YES BOGOTA 2026-2,B2G_YES BOGOTA-2 C1_SEP_2026,B2G Única,CC,1015455378,Abril Fonseca Manuel Fabian,2026-09-07,A,2026-10-21_1548_Inasistencias_por_curso_B2.1-7...
1,B2.1-7PM215,SDDE26-2_B2-1,Pedreros Cely Sergio Alejandro,YES BOGOTA 2026-2,B2G_YES BOGOTA-2 C1_SEP_2026,B2G Única,CC,1015455378,Abril Fonseca Manuel Fabian,2026-09-08,A,2026-10-21_1548_Inasistencias_por_curso_B2.1-7...
2,B2.1-7PM215,SDDE26-2_B2-1,Pedreros Cely Sergio Alejandro,YES BOGOTA 2026-2,B2G_YES BOGOTA-2 C1_SEP_2026,B2G Única,CC,1015455378,Abril Fonseca Manuel Fabian,2026-09-09,A,2026-10-21_1548_Inasistencias_por_curso_B2.1-7...
3,B2.1-7PM215,SDDE26-2_B2-1,Pedreros Cely Sergio Alejandro,YES BOGOTA 2026-2,B2G_YES BOGOTA-2 C1_SEP_2026,B2G Única,CC,1015455378,Abril Fonseca Manuel Fabian,2026-09-10,A,2026-10-21_1548_Inasistencias_por_curso_B2.1-7...
4,B2.1-7PM215,SDDE26-2_B2-1,Pedreros Cely Sergio Alejandro,YES BOGOTA 2026-2,B2G_YES BOGOTA-2 C1_SEP_2026,B2G Única,CC,1015455378,Abril Fonseca Manuel Fabian,2026-09-11,A,2026-10-21_1548_Inasistencias_por_curso_B2.1-7...


## Paso 9 · Validar antes de cargar

Es mejor que el error salga **aquí** y no en el reporte. Revisamos:

1. Que los estados sean solo `A`, `I` o `I*` (lo mismo que exige el `CHECK` de la tabla).
2. Que no haya filas repetidas para la misma clave primaria (curso + documento + fecha).
3. Que las fechas estén dentro del rango del curso.
4. Que **nuestros conteos coincidan** con `ASIST.ACUM` e `INAS.ACUM` del Excel.

In [14]:
def validar(df, df_acum, fecha_inicio, fecha_fin):
    errores = []

    # 1) estados válidos
    raros = set(df["estado"]) - {"A", "I", "I*"}
    if raros:
        errores.append(f"Estados no válidos: {raros}")

    # 2) duplicados en la clave primaria
    clave = ["curso_codigo", "tipo_documento", "numero_documento", "fecha"]
    n_dup = df.duplicated(clave).sum()
    if n_dup:
        errores.append(f"{n_dup} filas duplicadas en la clave")

    # 3) fechas en rango
    fuera = df[(df["fecha"] < fecha_inicio) | (df["fecha"] > fecha_fin)]
    if len(fuera):
        errores.append(f"{len(fuera)} marcas fuera del rango del curso")

    # 4) cuadre con los acumulados del Excel
    conteo = (df.assign(asis=df["estado"].eq("A"),
                        inas=df["estado"].str.startswith("I"))
                .groupby(["tipo_documento", "numero_documento"])[["asis", "inas"]].sum()
                .reset_index())
    cruce = df_acum.merge(conteo, how="left").fillna(0)
    no_cuadra = cruce[(cruce["asis"] != cruce["asist_acum"]) | (cruce["inas"] != cruce["inas_acum"])]
    if len(no_cuadra):
        errores.append(f"No cuadra con el Excel: {no_cuadra['numero_documento'].tolist()}")

    return errores

errores = validar(df, df_acum, fecha_inicio, fecha_fin)
print("✅ Todo en orden" if not errores else "❌ Problemas:\n- " + "\n- ".join(errores))
print("Estados encontrados:", df["estado"].value_counts().to_dict())

✅ Todo en orden
Estados encontrados: {'A': 743, 'I': 187}


## Paso 10 · Conectarse a SQL Server

`URL.create` arma la cadena de conexión sin que tengas que preocuparte por caracteres especiales en la clave.

- `fast_executemany=True` hace que la inserción de miles de filas sea **mucho** más rápida.
- `TrustServerCertificate=yes` evita el error de certificado típico en servidores locales.

La celda prueba la conexión y revisa que exista la tabla `dbo.asistencia`.

In [15]:
engine = None

if CARGAR_EN_BD:
    from sqlalchemy import create_engine, text
    from sqlalchemy.engine import URL

    parametros = {
        "driver": DRIVER,
        "Encrypt": "yes",
        "TrustServerCertificate": "yes"
    }

    url = URL.create(
        "mssql+pyodbc",
        username=USUARIO,
        password=CLAVE,
        host=SERVIDOR,
        database=BASE,
        query=parametros
    )
    engine = create_engine(url, fast_executemany=True)

    with engine.connect() as conn:
        print("Conectado a:", conn.execute(text("SELECT @@SERVERNAME")).scalar(), "| base:", BASE)
        existe = conn.execute(text("SELECT OBJECT_ID('dbo.asistencia', 'U')")).scalar()
        print("Tabla dbo.asistencia:", "✅ existe" if existe else "❌ no existe")
else:
    print("CARGAR_EN_BD = False -> no me conecto (modo prueba)")

Conectado a: bi-db-group | base: rendimiento_academico
Tabla dbo.asistencia: ✅ existe


## Paso 11 · Cargar con MERGE

Cada Excel es **acumulado**: el de la próxima semana trae otra vez las fechas que ya cargaste. Un `INSERT` normal chocaría con la clave primaria. Por eso usamos **`MERGE`**, que compara fila por fila:

| Situación | Qué hace |
|---|---|
| La fila **no existe** en la tabla | La **inserta** (`cargado_en` se llena solo por el DEFAULT) |
| Existe pero **cambió** (p. ej. una `I` pasó a `I*`) | La **actualiza** y pone `cargado_en = SYSDATETIME()` |
| Existe y es **igual** | No hace nada |

**Cómo lo hacemos:**
1. Subimos las filas a una **tabla temporal** `#stg` (el `#` significa que solo existe durante esta conexión).
2. Hacemos el `MERGE` de `#stg` contra `dbo.asistencia`.
3. Todo va dentro de `engine.begin()`, que es una **transacción**: si algo falla, no queda nada a medias.

In [16]:
COLUMNAS = ["curso_codigo", "asignatura", "docente", "programa", "periodo", "sede_jornada",
            "tipo_documento", "numero_documento", "nombre_completo", "fecha", "estado",
            "archivo_origen"]

SQL_CREAR_STG = '''
CREATE TABLE #stg (
    curso_codigo     VARCHAR(30)   NOT NULL,
    asignatura       NVARCHAR(40)  NOT NULL,
    docente          NVARCHAR(120) NULL,
    programa         NVARCHAR(100) NOT NULL,
    periodo          NVARCHAR(60)  NOT NULL,
    sede_jornada     NVARCHAR(60)  NOT NULL,
    tipo_documento   VARCHAR(5)    NOT NULL,
    numero_documento VARCHAR(20)   NOT NULL,
    nombre_completo  NVARCHAR(150) NOT NULL,
    fecha            DATE          NOT NULL,
    estado           VARCHAR(2)    NOT NULL,
    archivo_origen   NVARCHAR(255) NOT NULL
);
CREATE TABLE #acciones (accion NVARCHAR(10));
'''

SQL_MERGE = '''
MERGE dbo.asistencia AS t                 -- t = tabla destino
USING #stg AS s                           -- s = datos nuevos
   ON  t.curso_codigo     = s.curso_codigo
   AND t.tipo_documento   = s.tipo_documento
   AND t.numero_documento = s.numero_documento
   AND t.fecha            = s.fecha
WHEN MATCHED AND (   t.estado COLLATE Latin1_General_BIN2 <> s.estado
                  OR t.nombre_completo <> s.nombre_completo
                  OR ISNULL(t.docente, N'') <> ISNULL(s.docente, N'')
                  OR t.asignatura <> s.asignatura) THEN
    UPDATE SET t.estado          = s.estado,
               t.nombre_completo = s.nombre_completo,
               t.docente         = s.docente,
               t.asignatura      = s.asignatura,
               t.programa        = s.programa,
               t.periodo         = s.periodo,
               t.sede_jornada    = s.sede_jornada,
               t.archivo_origen  = s.archivo_origen,
               t.cargado_en      = SYSDATETIME()
WHEN NOT MATCHED BY TARGET THEN
    INSERT (curso_codigo, asignatura, docente, programa, periodo, sede_jornada,
            tipo_documento, numero_documento, nombre_completo, fecha, estado, archivo_origen)
    VALUES (s.curso_codigo, s.asignatura, s.docente, s.programa, s.periodo, s.sede_jornada,
            s.tipo_documento, s.numero_documento, s.nombre_completo, s.fecha, s.estado, s.archivo_origen)
OUTPUT $action INTO #acciones;            -- guarda si fue INSERT o UPDATE
'''

def cargar(df, engine):
    from sqlalchemy import text
    filas = df[COLUMNAS].to_dict("records")
    with engine.begin() as conn:                       # transacción: todo o nada
        conn.exec_driver_sql(SQL_CREAR_STG)
        conn.execute(text(f"INSERT INTO #stg ({', '.join(COLUMNAS)}) "
                          f"VALUES ({', '.join(':' + c for c in COLUMNAS)})"), filas)
        conn.exec_driver_sql(SQL_MERGE)
        resumen = dict(conn.exec_driver_sql(
            "SELECT accion, COUNT(*) FROM #acciones GROUP BY accion").fetchall())
        conn.exec_driver_sql("DROP TABLE #stg; DROP TABLE #acciones;")
    insertadas  = resumen.get("INSERT", 0)
    actualizadas = resumen.get("UPDATE", 0)
    return insertadas, actualizadas, len(filas) - insertadas - actualizadas

# Probamos con el archivo de los pasos anteriores
if engine is not None and not errores:
    ins, act, igual = cargar(df, engine)
    print(f"{codigo_curso}: {ins} insertadas, {act} actualizadas, {igual} sin cambios")
else:
    print("Modo prueba o hay errores de validación: no se cargó nada.")

B2.1-7PM215: 930 insertadas, 0 actualizadas, 0 sin cambios


## Paso 12 · Procesar todos los archivos

Juntamos los pasos 3 a 9 en una sola función `procesar_archivo()` y la aplicamos a cada Excel de la carpeta.

- Si un archivo tiene errores, se reporta y **se sigue con los demás**.
- Si `CARGAR_EN_BD = True`, cada archivo válido se carga con el `MERGE` del paso 11.
- Al final ves una tabla resumen.

Puedes ejecutar esta celda varias veces: gracias al `MERGE`, volver a cargar un archivo **no duplica nada**.

In [17]:
def procesar_archivo(ruta):
    # Pasos 3 a 8 en una sola función. Devuelve (df_largo, df_acum, inicio, fin)
    hoja = load_workbook(ruta, data_only=True, read_only=True).worksheets[0]
    filas = [list(f) for f in hoja.iter_rows(values_only=True)]

    meta = leer_metadatos(filas)
    curso = codigo_desde_nombre(ruta.name)

    i_enc = next(i for i, f in enumerate(filas) if any(limpiar_texto(c) == "Estudiantes" for c in f))
    enc = [limpiar_texto(c) for c in filas[i_enc]]
    c_nom, c_doc = enc.index("Estudiantes"), enc.index("Identificación")
    c_asis, c_inas = enc.index("ASIST.ACUM"), enc.index("INAS.ACUM")
    c_fechas = {j: f for j, h in enumerate(enc) if (f := fecha_encabezado(h))}

    largas, acum = [], []
    for fila in filas[i_enc + 1:]:
        nombre = limpiar_texto(fila[c_nom])
        if not nombre or nombre.upper() == "GLOSARIO":
            break
        tipo, numero = separar_documento(fila[c_doc])
        acum.append({"tipo_documento": tipo, "numero_documento": numero,
                     "asist_acum": fila[c_asis], "inas_acum": fila[c_inas]})
        for j, fecha in c_fechas.items():
            estado = (limpiar_texto(fila[j]) or "").upper()
            if estado:
                largas.append({"curso_codigo": curso, "asignatura": meta["asignatura"],
                               "docente": meta["docente"], "programa": meta["programa"],
                               "periodo": meta["periodo"], "sede_jornada": meta["sede_jornada"],
                               "tipo_documento": tipo, "numero_documento": numero,
                               "nombre_completo": nombre, "fecha": fecha, "estado": estado,
                               "archivo_origen": ruta.name})
    return (pd.DataFrame(largas), pd.DataFrame(acum),
            fecha_dia_mes_anio(meta["fecha_inicio"]), fecha_dia_mes_anio(meta["fecha_fin"]))


resumen, todos = [], []
for ruta in archivos:
    fila_resumen = {"archivo": ruta.name[:60]}
    try:
        df_i, acum_i, ini_i, fin_i = procesar_archivo(ruta)
        errs = validar(df_i, acum_i, ini_i, fin_i)
        fila_resumen.update(curso=df_i["curso_codigo"].iat[0],
                            estudiantes=df_i["numero_documento"].nunique(), filas=len(df_i))
        if errs:
            fila_resumen["resultado"] = "❌ " + " | ".join(errs)
        elif engine is not None:
            ins, act, igual = cargar(df_i, engine)
            fila_resumen["resultado"] = f"✅ {ins} insertadas, {act} actualizadas, {igual} sin cambios"
            todos.append(df_i)
        else:
            fila_resumen["resultado"] = "✅ válido (modo prueba, no se cargó)"
            todos.append(df_i)
    except Exception as e:
        fila_resumen["resultado"] = f"❌ {e}"
    resumen.append(fila_resumen)

pd.DataFrame(resumen)

,archivo,curso,estudiantes,filas,resultado
0,2026-10-21_1548_Inasistencias_por_curso_B2.1-7...,B2.1-7PM215,30,930,"✅ 0 insertadas, 0 actualizadas, 930 sin cambios"
1,2026-10-21_1548_Inasistencias_por_curso_B2.1-7...,B2.1-7PM216,30,930,"✅ 930 insertadas, 0 actualizadas, 0 sin cambios"
2,2026-10-21_1548_Inasistencias_por_curso_C1.1-7...,C1.1-7PM217,30,930,"✅ 930 insertadas, 0 actualizadas, 0 sin cambios"
3,2026-10-21_1549_Inasistencias_por_curso_B2.1-6...,B2.1-6AM60,30,960,"✅ 960 insertadas, 0 actualizadas, 0 sin cambios"
4,2026-10-21_1549_Inasistencias_por_curso_B2.1-7...,B2.1-7PM205,30,930,"✅ 930 insertadas, 0 actualizadas, 0 sin cambios"
5,2026-10-21_1549_Inasistencias_por_curso_B2.1-7...,B2.1-7PM206,30,930,"✅ 930 insertadas, 0 actualizadas, 0 sin cambios"
6,2026-10-21_1549_Inasistencias_por_curso_B2.1-7...,B2.1-7PM207,30,930,"✅ 930 insertadas, 0 actualizadas, 0 sin cambios"
7,2026-10-21_1549_Inasistencias_por_curso_B2.1-7...,B2.1-7PM208,30,930,"✅ 930 insertadas, 0 actualizadas, 0 sin cambios"
8,2026-10-21_1549_Inasistencias_por_curso_B2.1-7...,B2.1-7PM209,30,930,"✅ 930 insertadas, 0 actualizadas, 0 sin cambios"
9,2026-10-21_1549_Inasistencias_por_curso_B2.1-7...,B2.1-7PM210,30,930,"✅ 930 insertadas, 0 actualizadas, 0 sin cambios"


In [ ]:
# Opcional: guardar los datos limpios en CSV para revisarlos en Excel
if todos:
    pd.concat(todos).to_csv(Path(CARPETA) / "asistencia_limpia.csv",
                            index=False, encoding="utf-8-sig")   # utf-8-sig = tildes bien en Excel
    print("CSV guardado en:", Path(CARPETA) / "asistencia_limpia.csv")

## Paso 13 · Verificar en la base de datos

Contamos filas y estudiantes por curso **directamente en SQL Server**. Deberías ver unas 840 a 870 filas y 30 estudiantes por curso.

In [43]:
if engine is not None:
    consulta = '''
        SELECT curso_codigo,
               COUNT(*)                         AS filas,
               COUNT(DISTINCT numero_documento) AS estudiantes,
               SUM(CASE WHEN estado LIKE 'I%' THEN 1 ELSE 0 END) AS inasistencias,
               MAX(cargado_en)                  AS ultima_carga
        FROM dbo.asistencia
        GROUP BY curso_codigo
        ORDER BY curso_codigo
    '''
    display(pd.read_sql(consulta, engine))
else:
    print("Modo prueba: cambia CARGAR_EN_BD = True en el paso 1 para cargar y verificar.")

,curso_codigo,filas,estudiantes,inasistencias,ultima_carga
0,B1.1-5PM100,840,30,289,2026-10-16 10:40:10
1,B1.1-5PM101,840,30,147,2026-10-16 10:40:10
2,B1.1-5PM102,840,30,213,2026-10-16 10:40:10
3,B1.1-5PM103,840,30,212,2026-10-16 10:40:10
4,B1.1-5PM104,840,30,338,2026-10-16 10:40:11
...,...,...,...,...,...
306,B2.1-8AM54,338,26,73,2026-10-16 10:40:12
307,B2.1-8AM55,420,30,46,2026-10-16 10:40:12
308,B2.1-8AM56,420,30,161,2026-10-16 10:40:13
309,C1.1-7PM116,390,30,153,2026-10-16 10:40:13


---
### Resumen de lo que aprendiste

- **Buscar por etiqueta y no por celda fija** hace que el código aguante cambios en la plantilla.
- **El formato ancho del Excel se pasa a largo** (una fila por fecha) antes de guardarlo.
- **Una celda vacía no es una falta**: es una clase aún no registrada.
- **Validar antes de cargar**, incluyendo el cuadre con los totales del propio Excel.
- **`MERGE` + tabla temporal + transacción** = cargas repetibles sin duplicados.